# Tarea 2: Fundamentos de Python
## Ciencia de Datos Ambientales - UTEC

**Nombre:** Jessica Elizabeth Yovera Ramos
**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa todos los problemas en este notebook
- Escribe tu codigo en las celdas proporcionadas
- Ejecuta todas las celdas antes de entregar
- Sube el archivo `.ipynb` completado al modulo correspondiente en Canvas

**Integridad academica:** Tarea individual. Puedes consultar materiales del curso y documentacion de Python, pero todo el codigo debe ser tuyo.

---

## Problema 1: Procesador de Nombres de Archivos Landsat (10 puntos)

Trabajas con imagenes satelitales Landsat del Peru. Los nombres siguen el formato:

```
LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
```
Componentes: `{sensor}_{nivel}_{path_row}_{fecha}_{coleccion}_{tier}_SR_{banda}.TIF`

> Los paths 003-009, filas 062-071 cubren el territorio peruano (Madre de Dios, Loreto, Lima, Cusco).

### Tus tareas:

**Parte A (4 pts):** Funcion `procesar_nombre_landsat(nombre_archivo)` que devuelva un diccionario con:
- `sensor` (ej. "LC08"), `path` (ej. "008"), `row` (ej. "067")
- `fecha` formateada como "AAAA-MM-DD"
- `banda` (ej. "B4")

**Parte B (3 pts):** Funcion `clasificar_banda(banda)` que devuelva el nombre segun la tabla:

| Banda | Nombre |
|-------|--------|
| B1 | Aerosol costero | B2 | Azul | B3 | Verde | B4 | Rojo |
| B5 | Infrarrojo cercano (NIR) | B6 | SWIR1 | B7 | SWIR2 |

Si no esta en la tabla, devuelve "Desconocida".

**Parte C (3 pts):** Procesa la lista de archivos: parsea, imprime resumen (fecha/path/row/banda) y cuenta cuantas fechas unicas hay.

In [ ]:
# Archivos Landsat sobre el Peru (paths 008-009: Madre de Dios, Ucayali, Loreto)
archivos = [
    "LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF",
    "LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF",
    "LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF",
    "LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B4.TIF",
]

# Parte A: funcion procesar_nombre_landsat
def procesar_nombre_landsat(nombre_archivo):
  partes = nombre_archivo.replace(".TIF", "").split("_")
  path_row = partes[2]
  fecha_original = partes[3]

  return {
    "sensor": partes[0],
    "path": path_row[:3],
    "row": path_row[3:],
    "fecha": f"{fecha_original[:4]}-{fecha_original[4:6]}-{fecha_original[6:]}",
    "banda": partes[-1]
    }


# Parte B: funcion clasificar_banda
def clasificar_banda(banda):
    bandas = {
        "B1": "Aerosol costero",
        "B2": "Azul",
        "B3": "Verde",
        "B4": "Rojo",
        "B5": "Infrarrojo cercano (NIR)",
        "B6": "SWIR1",
        "B7": "SWIR2"
    }

    return bandas.get(banda, "Desconocida")

# Parte C: procesar todos los archivos
fechas = []

for archivo in archivos:
    datos = procesar_nombre_landsat(archivo)
    print(datos["fecha"], datos["path"], datos["row"],datos["banda"], clasificar_banda(datos['banda']))
    fechas.append(datos["fecha"])

print("Fechas unicas:", len(set(fechas)))


2024-06-15 008 067 B4 Rojo
2024-06-15 008 067 B5 Infrarrojo cercano (NIR)
2024-07-01 008 067 B3 Verde
2024-06-15 009 067 B4 Rojo
2024-07-08 008 067 B6 SWIR1
2024-07-01 008 067 B4 Rojo
Fechas unicas: 3


---
## Problema 2: Inventario Forestal en Madre de Dios (10 puntos)

El **SERFOR** realiza inventarios forestales en Madre de Dios. Los datos incluyen valores faltantes (`-999`) y mediciones con posibles errores.

**Parte A (3 pts):** Funcion `calcular_area_basal(dap_cm)`:
- Devuelve AB en m2: $AB = \pi 	\times  (DAP/200)^2$
- Devuelve `None` si DAP <= 0 o == -999

**Parte B (3 pts):** Funcion `clasificar_arbol(dap_cm, altura_m)` que devuelva:
- `clase`: "Brinzal" (<10cm), "Latizal" (10-25cm), "Fustal menor" (25-50cm), "Fustal mayor" (>=50cm)
- `alerta`: True si DAP > 200cm, altura > 60m, o altura < 1m con DAP > 10cm

**Parte C (4 pts):** Procesa los datos:
1. Para cada árbol, calcule el área basal y clasifíquelo.
2. Omita los árboles con datos faltantes (valores -999).
3. Imprima una advertencia para los árboles marcados.
4. Calcule e imprima las estadísticas descriptivas:
- Número total de árboles válidos
- Área basal total (suma de todos los árboles válidos)
- Cantidad de árboles en cada clase de tamaño
- Número de registros marcados

In [1]:
import math
# Inventario forestal - Madre de Dios, Peru (datos SERFOR)
# Formato: [id, especie, dap_cm, altura_m]
datos_arboles = [
    [1,  "Swietenia macrophylla",      35.4, 22.1],   # Caoba
    [2,  "Cedrela odorata",            28.2, 18.5],   # Cedro
    [3,  "Cedrelinga cateniformis",   -999,  25.0],   # Tornillo - DAP faltante
    [4,  "Virola surinamensis",        18.7, 12.3],   # Cumala
    [5,  "Dipteryx micrantha",         52.1, 24.8],   # Shihuahuaco
    [6,  "Calycophyllum spruceanum",    8.5,  6.2],   # Capirona
    [7,  "Terminalia oblonga",         45.0, 85.0],   # Yacushapana - altura sospechosa
    [8,  "Cedrelinga cateniformis",    62.3, 28.4],   # Tornillo
    [9,  "Swietenia macrophylla",      41.2, -999],   # Caoba - altura faltante
    [10, "Hura crepitans",             22.5,  0.5],   # Catahua - sospechoso
    [11, "Schizolobium parahybum",      5.2,  3.1],   # Pino chuncho
    [12, "Guazuma crinita",            38.9, 21.7],   # Bolaina
]

# Parte A: Escribe la función calcular_area_basal aqui:
def calcular_area_basal(dap_cm):
    if dap_cm <= 0 or dap_cm == -999:
        return None
    return math.pi * (dap_cm / 200) ** 2

# Parte B: Escribe la función clasificar_arbol aquí:
def clasificar_arbol(dap_cm, altura_m):
    if dap_cm < 10:
        clase = "Brinzal"
    elif dap_cm < 25:
        clase = "Latizal"
    elif dap_cm < 50:
        clase = "Fustal menor"
    else:
        clase = "Fustal mayor"

    alerta = (
        dap_cm > 200
        or altura_m > 60
        or (altura_m < 1 and dap_cm > 10)
    )
    return {
        "clase": clase,
        "alerta": alerta
    }

# Parte C: Procesar los datos e imprimir los resultados
area_basal_total = 0
arboles_validos = 0
registros_marcados = 0

clases = {
    "Brinzal": 0,
    "Latizal": 0,
    "Fustal menor": 0,
    "Fustal mayor": 0
}
for arbol in datos_arboles:
    id_arbol, especie, dap, altura = arbol
    if dap == -999 or altura == -999:
        continue

    area_basal = calcular_area_basal(dap)
    clasificacion = clasificar_arbol(dap, altura)
    arboles_validos += 1
    area_basal_total += area_basal
    clases[clasificacion["clase"]] += 1

    if clasificacion["alerta"]:
        registros_marcados += 1
        print(
            f"ADVERTENCIA: Arbol {id_arbol} - {especie}  ", f"DAP: {dap} cm  Altura: {altura} m"
        )

print("\n ESTADISTICAS DEL INVENTARIO ")
print(f"Arboles validos: {arboles_validos}")
print(f"Area basal total: {area_basal_total:.4f} m2")
print("\nCantidad de arboles por clase:")

for clase, cantidad in clases.items():
    print(f"- {clase}: {cantidad}")

print(f"\nRegistros marcados: {registros_marcados}")

ADVERTENCIA: Arbol 7 - Terminalia oblonga   DAP: 45.0 cm  Altura: 85.0 m
ADVERTENCIA: Arbol 10 - Hura crepitans   DAP: 22.5 cm  Altura: 0.5 m

 ESTADISTICAS DEL INVENTARIO 
Arboles validos: 10
Area basal total: 1.0318 m2

Cantidad de arboles por clase:
- Brinzal: 2
- Latizal: 2
- Fustal menor: 4
- Fustal mayor: 2

Registros marcados: 2


---
## Lista de verificacion
- [ ] Todas las celdas corren sin errores
- [ ] Ambos problemas estan completos
- [ ] Salidas visibles en todas las celdas
- [ ] Nombre incluido